# 📖 Notebook 5: Sessions, Watches & When Not to Use ZooKeeper

The first four notebooks showed you **what ZooKeeper can do**: locks, leader
election, config, service discovery. This notebook zooms out and explains **the
hidden rules** that make all of those things work (or fail):

1. **Sessions** — the lifeline between a client and the ensemble.
2. **Watches** — the only way ZooKeeper pushes events to you, and their surprising
   limits.
3. **ZAB / quorum** — why ZooKeeper ensembles have 3 or 5 nodes (never 4).
4. **When NOT to use ZooKeeper** — because using it wrong is worse than not
   using it at all.

> This notebook is mostly conceptual. The code cells are short and use ZooKeeper
> to *observe* these behaviors rather than build a product feature.

## Learning Objectives

- Tell the difference between `CONNECTED`, `SUSPENDED`, and `LOST` — and what a
  well-behaved app should do in each state.
- Explain why ephemeral nodes are tied to *session expiration*, not to a
  momentary disconnect.
- List the three surprising things about ZooKeeper watches.
- Explain why a 5-node cluster survives 2 failures but a 4-node cluster only
  survives 1.
- Know when to pick something else (Redis, etcd, a real DB, Kafka).


## 🛠️ Setup

```bash
cd 03-technologies/coordination/zookeeper
docker compose up -d
```

Select the `.venv` kernel (top-right). Reload the VS Code window if it's not
listed.


In [ ]:
import time
from kazoo.client import KazooClient
from kazoo.protocol.states import KazooState


---
## 1. Sessions: the Lifeline

When a kazoo client calls `zk.start()`, it opens a **session** with the
ensemble. Everything important in ZooKeeper — ephemeral nodes, watches, locks —
is tied to that session.

The session moves through three states:

| State | Meaning | Are ephemeral nodes still there? |
|-------|---------|----------------------------------|
| `CONNECTED` | Talking to a ZK server normally | ✅ Yes |
| `SUSPENDED` | Lost the TCP connection, trying to reconnect | ✅ Yes (grace period) |
| `LOST` | Session **expired** on the server side | ❌ Gone — server deleted them |

**The critical rule**: ephemeral nodes don't vanish the moment your network
hiccups. They vanish when the **server** declares your session expired, after
the session timeout (often 10–30 seconds).

This is great — it means a 2-second network blip doesn't trigger a leader
election. But it has a dangerous consequence for *you, the application*.


In [ ]:
# Observe the state transitions of a kazoo client
zk = KazooClient(hosts="localhost:2181,localhost:2182,localhost:2183")

state_history = []

def on_state(state):
    state_history.append((time.time(), state))
    print(f"  kazoo state -> {state}")

zk.add_listener(on_state)
zk.start()
time.sleep(0.5)
print(f"Current state: {zk.state}")
print(f"Session timeout negotiated with server: {zk.client_state}")


### ⚠️ The Dangerous Misconception: "I'm still the leader because I still have my node"

If your connection is `SUSPENDED`:

- Your ephemeral node **might still exist** on the server.
- But **another client might have already been told it's the new leader** —
  because from *its* point of view, your session is about to expire.

So: a `SUSPENDED` process that keeps acting as leader can end up in
**split-brain** with the real, newly-elected leader.

**The rule of thumb every distributed lock / leader application must follow:**

> On `SUSPENDED`, *pause* any exclusive work. On `LOST`, *assume you no longer
> own anything* — rejoin the election, re-acquire the lock, re-register the
> service.

kazoo gives you this hook via `add_listener`:


In [ ]:
# Example skeleton — NOT run against a real partition here,
# but this is the pattern you should always follow.

is_safe_to_work = False

def my_state_handler(state):
    global is_safe_to_work
    if state == KazooState.LOST:
        # Session expired: ephemeral nodes gone, locks/leadership lost
        is_safe_to_work = False
        print("  🔴 LOST: stop all exclusive work, re-join election on reconnect")
    elif state == KazooState.SUSPENDED:
        # Network blip: play it safe, don't make exclusive decisions
        is_safe_to_work = False
        print("  🟡 SUSPENDED: pause writes / leader duties until reconnected")
    elif state == KazooState.CONNECTED:
        is_safe_to_work = True
        print("  🟢 CONNECTED: safe to proceed (re-verify leadership first!)")

zk.add_listener(my_state_handler)
print("Installed state handler. In a real app, all 'am I the leader?' logic")
print("would be gated by is_safe_to_work plus a re-check of the ZK state.")


---
## 2. Watches: the Three Rules You Must Internalize

We've been using watches since notebook 1. Here are the three things about them
that beginners get wrong.

### Rule 1 — Raw watches are one-shot

In the **low-level** API, a watch fires *once* and is then removed. If you want
continuous notifications, you re-register it every time.

Good news: `kazoo`'s `DataWatch` and `ChildrenWatch` decorators re-register
automatically. Bad news: if you ever drop to the low-level API, you need to
remember this.

### Rule 2 — Events are thin — they don't carry the new value

A `DataWatch` event is a nudge, not a payload. It says *"the thing changed"*,
not *"here's what it changed to"*. Your callback must **re-read** the current
state. kazoo does this for you (it reads the value and passes it as `data`),
but the raw ZooKeeper event has no payload.

### Rule 3 — Multiple changes can collapse into one notification

If a znode changes 10 times rapidly before your watch fires, you will see **one**
notification, and when you re-read you'll see the *latest* value. You will *not*
see all 10 intermediate values.

**Conclusion:** ZooKeeper watches are "refresh your view" signals, not an event
log. Never use them as a message queue. For ordered, durable events, use Kafka.


In [ ]:
# Demo Rule 3: rapid writes collapse into fewer notifications
zk.ensure_path("/demo/watch-rules")
zk.set("/demo/watch-rules", b"0")

notifications = []

@zk.DataWatch("/demo/watch-rules")
def on_change(data, stat):
    notifications.append((time.time(), data.decode() if data else None))

time.sleep(0.3)  # let the initial event fire
print(f"Initial notifications: {len(notifications)}")

# Rapid-fire 20 writes
for i in range(1, 21):
    zk.set("/demo/watch-rules", str(i).encode())

time.sleep(1)  # let watches settle
print(f"Total notifications after 20 writes: {len(notifications)}")
print(f"Values observed: {[n[1] for n in notifications]}")
print()
print("👉 You probably saw *fewer* than 20 notifications, and the last value")
print("   you saw is '20'. Intermediate values were merged away.")
print("   Watches are not an audit log.")


---
## 3. Why Ensembles Have 3 or 5 Nodes — Never 4

ZooKeeper uses a consensus protocol called **ZAB** (ZooKeeper Atomic Broadcast).
To accept a write, a **majority (quorum) of nodes must agree**. This is how ZK
avoids split-brain: at most one partition can contain a majority.

| Ensemble size | Majority needed | Failures tolerated |
|---------------|-----------------|--------------------|
| 1 | 1 | 0 (any failure = outage) |
| 3 | 2 | 1 |
| 4 | 3 | 1 (same as 3 — but more expensive!) |
| 5 | 3 | 2 |
| 7 | 4 | 3 |

**Key insights:**

- **Odd numbers are optimal.** A 4-node cluster costs 33% more than 3 but tolerates
  the same number of failures.
- **If you lose a majority, all writes stop.** Reads may still work against a
  minority (with stale data if you're using non-synced reads), but writes can't
  proceed. This is a deliberate choice: ZK picks **consistency** over
  **availability** (CP in CAP).
- **Geographical distribution is risky.** A 3-node ensemble split across 3
  regions can be killed by a single inter-region network partition. Most
  production setups place ZK within one region / one availability zone group.


In [ ]:
# Ask each ZK node directly for its role using the `mntr` four-letter word.
# This is a fun way to SEE the ensemble. Requires ZK's 4lw whitelist to allow 'mntr'.

import socket

def four_letter_word(host, port, word):
    try:
        with socket.create_connection((host, port), timeout=2) as s:
            s.sendall(word.encode())
            data = b""
            while True:
                chunk = s.recv(4096)
                if not chunk:
                    break
                data += chunk
            return data.decode(errors="replace")
    except Exception as e:
        return f"(not reachable: {e})"

for port in (2181, 2182, 2183):
    print(f"=== localhost:{port} ===")
    # 'srvr' is almost always allowed; 'mntr' is more verbose but often blocked
    out = four_letter_word("localhost", port, "srvr")
    for line in out.splitlines():
        if line.startswith(("Mode:", "Zookeeper version:", "Node count:")):
            print(f"  {line}")
    print()
print("👉 Exactly one node will report 'Mode: leader'. The others are 'follower'.")
print("   This is the ZAB-elected internal leader — not to be confused with")
print("   *your* leader election in notebook 2.")


---
## 4. When NOT to Use ZooKeeper

ZooKeeper is a beautifully specialized tool. Using it for the wrong job is
painful.

### ❌ Don't use ZooKeeper as a database

- Each znode's value is capped at **~1 MB** (1,048,576 bytes by default).
- ZK is optimized for small metadata, not bulk data.
- All data lives in memory on every server.
- **Use instead:** PostgreSQL, DynamoDB, S3, or whatever fits your data shape.

### ❌ Don't use ZooKeeper as a message queue or event log

- Watches drop intermediate events (see Rule 3 above).
- There is no consumer-offset / replay mechanism.
- **Use instead:** Kafka, RabbitMQ, Redis Streams.

### ❌ Don't use ZooKeeper for high write throughput

- Every write goes through ZAB consensus → every write hits the leader.
- Typical ceiling: **a few thousand writes/sec** across the whole ensemble.
- Reads scale better (they can be served from followers).
- **Use instead:** Redis, Cassandra, or any sharded store.

### ❌ Don't use ZooKeeper for cross-region coordination (usually)

- ZAB needs low-latency majority commits.
- Cross-region latency can turn a 10 ms write into a 200 ms write.
- **Use instead:** regional ZK ensembles with application-level reconciliation,
  or a WAN-aware system like CockroachDB / Spanner.

### ✅ Do use ZooKeeper for

- Leader election
- Distributed locks
- Service discovery (membership)
- Small, live-updating configuration
- Cluster metadata for another system (Kafka, HBase, Solr)

### Modern alternatives you should know

- **etcd** — similar use cases, Raft-based, HTTP+gRPC API, used by Kubernetes.
- **Consul** — service discovery + KV + health checks, more ops-friendly.
- **Chubby** (Google, internal only) — ZooKeeper's spiritual predecessor.
- **KRaft** — Kafka's built-in Raft controller, which lets Kafka drop ZooKeeper
  entirely. A good sign of how the industry is evolving: where a full external
  coordinator used to be required, Raft-based embedded quorums are increasingly
  the pattern.


---
## 📊 Summary

- Everything in ZooKeeper — ephemeral nodes, locks, leadership, watches — hangs
  off the **session**. Understand `CONNECTED` / `SUSPENDED` / `LOST` and your
  app will be boring. Ignore them and you will have split-brain bugs that only
  reproduce in production.
- **Watches are nudges**, not events. Re-read state after every notification.
  They collapse under load and carry no payload.
- **Majorities** are why ZK ensembles are sized 3, 5, 7. The CAP tradeoff is
  *intentional*: ZooKeeper prefers being correct to being available.
- ZooKeeper is a great **coordinator**. It is a terrible **database**, message
  queue, or high-throughput store.

### Congratulations

You've completed the ZooKeeper lab. You can now:

- Build a distributed lock (notebook 1).
- Implement leader election with automatic failover (notebook 2).
- Push live config to any number of servers (notebook 3).
- Run a service registry with automatic crash detection (notebook 4).
- Reason about when ZooKeeper is the right — or wrong — tool (this notebook).

A great next step is to read the ZooKeeper recipes in the official docs:
<https://zookeeper.apache.org/doc/current/recipes.html>


In [ ]:
# Cleanup
try:
    if zk.exists("/demo"):
        zk.delete("/demo", recursive=True)
finally:
    zk.stop()
print("Done.")
